# Uruti.Rw Domain-Specific Startup Advisory Chatbot

## Project Overview

This notebook implements a domain-specific chatbot that integrates with the Uruti.Rw MLOps platform for startup classification. The chatbot provides personalized advice based on three categories:
- **Mentorship Needed**: Guidance for early-stage entrepreneurs
- **Investment Ready**: Advice for scaling and fundraising
- **Needs Refinement**: Support for addressing business challenges

## 1. Environment Setup and Dependencies

In [2]:
!pip install transformers torch datasets evaluate accelerate gradio
!pip install scikit-learn pandas numpy matplotlib seaborn
!pip install huggingface_hub tokenizers

import json
import pandas as pd
import numpy as np
import torch
import matplotlib.pyplot as plt
#import seaborn as sns
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

In [3]:
# Check if CUDA is available
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

Using device: cpu


## 2. Dataset Loading and Analysis

In [4]:
import os
print(f"Working directory: {os.getcwd()}")

Working directory: /Users/davidniyonshutii/Downloads/uruti_MLOP/Domain-specific


In [5]:
# Load the comprehensive startup advisory dataset
# check if the data directory exists
if not os.path.exists('data'):
    raise FileNotFoundError("The 'data' directory does not exist. Please ensure the dataset is available.")
with open('data/train_data.json', 'r') as f:
    train_data = json.load(f)

with open('data/val_data.json', 'r') as f:
    val_data = json.load(f)

with open('data/test_data.json', 'r') as f:
    test_data = json.load(f)

print(f"Dataset sizes:")
print(f"  Training: {len(train_data)} examples")
print(f"  Validation: {len(val_data)} examples")
print(f"  Test: {len(test_data)} examples")

Dataset sizes:
  Training: 96 examples
  Validation: 12 examples
  Test: 12 examples


In [6]:
# Analyze dataset characteristics
def analyze_dataset(data, name):
    print(f"\n=== {name} Dataset Analysis ===")

    # Category distribution
    categories = {}
    response_lengths = []
    question_lengths = []

    for item in data:
        category = item['category']
        categories[category] = categories.get(category, 0) + 1
        response_lengths.append(len(item['output']))
        question_lengths.append(len(item['instruction']))

    print("Category Distribution:")
    for category, count in categories.items():
        print(f"  {category}: {count} ({count/len(data)*100:.1f}%)")

    print(f"\nText Statistics:")
    print(f"  Avg response length: {np.mean(response_lengths):.0f} chars")
    print(f"  Avg question length: {np.mean(question_lengths):.0f} chars")
    print(f"  Response length range: {min(response_lengths)}-{max(response_lengths)} chars")

analyze_dataset(train_data, "Training")
analyze_dataset(val_data, "Validation")
analyze_dataset(test_data, "Test")


=== Training Dataset Analysis ===
Category Distribution:
  mentorship_needed: 32 (33.3%)
  needs_refinement: 29 (30.2%)
  investment_ready: 35 (36.5%)

Text Statistics:
  Avg response length: 293 chars
  Avg question length: 48 chars
  Response length range: 143-753 chars

=== Validation Dataset Analysis ===
Category Distribution:
  needs_refinement: 6 (50.0%)
  investment_ready: 2 (16.7%)
  mentorship_needed: 4 (33.3%)

Text Statistics:
  Avg response length: 282 chars
  Avg question length: 52 chars
  Response length range: 149-682 chars

=== Test Dataset Analysis ===
Category Distribution:
  mentorship_needed: 6 (50.0%)
  investment_ready: 3 (25.0%)
  needs_refinement: 3 (25.0%)

Text Statistics:
  Avg response length: 241 chars
  Avg question length: 44 chars
  Response length range: 143-754 chars


## 3. Data Preprocessing and Tokenization

In [7]:
from transformers import AutoTokenizer, AutoModelForCausalLM
from datasets import Dataset

# Choose model - using DialoGPT for conversational fine-tuning
MODEL_NAME = "microsoft/DialoGPT-medium"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

# Add padding token if it doesn't exist
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print(f"Tokenizer vocabulary size: {tokenizer.vocab_size}")
print(f"Max position embeddings: {tokenizer.model_max_length}")

def preprocess_conversation_data(data, tokenizer, max_length=512):
    """
    Preprocess conversational data for DialoGPT training
    Format: [User] Question [Bot] Response [eos_token]
    """
    processed_data = {
        'input_ids': [],
        'attention_mask': [],
        'labels': []
    }

    for item in data:
        # Format conversation with special tokens
        conversation = f"[User] {item['instruction']} [Bot] {item['output']}"

        # Tokenize
        encoded = tokenizer(
            conversation,
            truncation=True,
            max_length=max_length,
            padding='max_length',
            return_tensors='pt'
        )

        processed_data['input_ids'].append(encoded['input_ids'].squeeze())
        processed_data['attention_mask'].append(encoded['attention_mask'].squeeze())
        # For language modeling, labels are the same as input_ids
        processed_data['labels'].append(encoded['input_ids'].squeeze())

    return processed_data

Tokenizer vocabulary size: 50257
Max position embeddings: 1024


In [8]:
# Preprocess datasets
train_processed = preprocess_conversation_data(train_data, tokenizer)
val_processed = preprocess_conversation_data(val_data, tokenizer)
test_processed = preprocess_conversation_data(test_data, tokenizer)

In [9]:
# Convert to HuggingFace datasets
train_dataset = Dataset.from_dict(train_processed)
val_dataset = Dataset.from_dict(val_processed)
test_dataset = Dataset.from_dict(test_processed)
print(f"Training examples: {len(train_dataset)}")
print(f"Validation examples: {len(val_dataset)}")
print(f"Test examples: {len(test_dataset)}")

Training examples: 96
Validation examples: 12
Test examples: 12


In [10]:
# Sample tokenized data
sample_input = train_dataset[0]['input_ids']
sample_text = tokenizer.decode(sample_input, skip_special_tokens=False)
print(f"\nSample tokenized conversation:")
print(f"Tokens: {len(sample_input)}")
print(f"Text: {sample_text[:200]}...")


Sample tokenized conversation:
Tokens: 512
Text: [User] What legal considerations should I be aware of? [Bot] Register your business entity, understand tax obligations, protect intellectual property, draft founder agreements, and comply with industr...


## 4. Model Architecture and Configuration

In [ ]:
from transformers import AutoModelForCausalLM, TrainingArguments, Trainer
from transformers import DataCollatorForLanguageModeling

# Load pre-trained DialoGPT model
model = AutoModelForCausalLM.from_pretrained(MODEL_NAME)
model = model.to(device)

print(f"Model parameters: {model.num_parameters():,}")
print(f"Model architecture: {model.config}")

# Data collator for language modeling
data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False,  # Causal LM, not masked LM
    pad_to_multiple_of=8
)

# Training arguments with hyperparameter optimization
training_args = TrainingArguments(
    output_dir='./startup-chatbot-model',
    overwrite_output_dir=True,

    # Training hyperparameters
    num_train_epochs=100,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    gradient_accumulation_steps=2,

    # Optimization
    learning_rate=5e-5,
    weight_decay=0.01,
    warmup_steps=100,
    adam_epsilon=1e-8,
    max_grad_norm=1.0,

    # Evaluation and logging
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_steps=10,
    save_steps=500,
    eval_steps=500,

    # Model selection
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,

    # Technical settings
    fp16=torch.cuda.is_available(),  # Use mixed precision
    dataloader_num_workers=0,
    remove_unused_columns=False,

    # Reproducibility
    seed=42,
    report_to=None  # Disable wandb/tensorboard for this demo
)

print(f"Training configuration:")
print(f"  Epochs: {training_args.num_train_epochs}")
print(f"  Batch size: {training_args.per_device_train_batch_size}")
print(f"  Learning rate: {training_args.learning_rate}")
print(f"  Mixed precision: {training_args.fp16}")

## 5. Model Training with Evaluation Metrics

In [ ]:
# install evaluate
!pip install evaluate

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


In [ ]:
from sklearn.metrics import accuracy_score, precision_recall_fscore_support
# Check if evaluate is installed
try:
    import evaluate
except ImportError:
    print("evaluate is not installed. Installing...")
    !pip install evaluate
import evaluate

# Load evaluation metrics
perplexity_metric = evaluate.load("perplexity", module_type="metric")

def compute_metrics(eval_pred):
    """Compute perplexity and other metrics for evaluation"""
    predictions, labels = eval_pred

    # Compute perplexity
    perplexity = perplexity_metric.compute(
        predictions=predictions,
        model_id=MODEL_NAME
    )

    return {
        "perplexity": perplexity["perplexity"]
    }

# Custom trainer class for better logging
class StartupChatbotTrainer(Trainer):
    def __init__(self, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self.train_losses = []
        self.eval_losses = []
        self.eval_perplexities = []

    def log(self, logs):
        super().log(logs)

        if "train_loss" in logs:
            self.train_losses.append(logs["train_loss"])

        if "eval_loss" in logs:
            self.eval_losses.append(logs["eval_loss"])

        if "eval_perplexity" in logs:
            self.eval_perplexities.append(logs["eval_perplexity"])

# Initialize trainer
trainer = StartupChatbotTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    data_collator=data_collator,
    tokenizer=tokenizer,
    compute_metrics=compute_metrics
)

print("Starting model fine-tuning...")
print(f"Training on {len(train_dataset)} examples")
print(f"Validating on {len(val_dataset)} examples")

# Train the model
train_result = trainer.train()

print(f"Training completed!")
print(f"Final training loss: {train_result.training_loss:.4f}")
print(f"Training time: {train_result.metrics['train_runtime']:.2f} seconds")

# Save the fine-tuned model
trainer.save_model('./startup-chatbot-model')
tokenizer.save_pretrained('./startup-chatbot-model')

print("Model saved to './startup-chatbot-model'")

evaluate is not installed. Installing...


huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


ModuleNotFoundError: No module named 'evaluate'

## 6. Model Evaluation and Performance Analysis

In [ ]:
# Evaluate on test set
test_results = trainer.evaluate(test_dataset)

print(f"=== Test Set Evaluation ===")
print(f"Test Loss: {test_results['eval_loss']:.4f}")
print(f"Test Perplexity: {test_results['eval_perplexity']:.4f}")

# Plot training curves
plt.figure(figsize=(15, 5))

# Training loss
plt.subplot(1, 3, 1)
plt.plot(trainer.train_losses, label='Training Loss')
plt.title('Training Loss Over Time')
plt.xlabel('Steps')
plt.ylabel('Loss')
plt.legend()
plt.grid(True)

# Validation loss
plt.subplot(1, 3, 2)
plt.plot(trainer.eval_losses, label='Validation Loss', color='orange')
plt.title('Validation Loss Over Time')
plt.xlabel('Epochs')
plt.ylabel('Loss')
plt.legend()
plt.grid(True)

# Perplexity
plt.subplot(1, 3, 3)
plt.plot(trainer.eval_perplexities, label='Validation Perplexity', color='green')
plt.title('Validation Perplexity Over Time')
plt.xlabel('Epochs')
plt.ylabel('Perplexity')
plt.legend()
plt.grid(True)

plt.tight_layout()
plt.show()

In [ ]:
# Qualitative evaluation - generate responses for sample questions
def generate_response(model, tokenizer, question, max_length=200, temperature=0.7, top_p=0.9):
    """Generate response to a startup advice question"""
    prompt = f"[User] {question} [Bot]"

    # Tokenize input
    inputs = tokenizer.encode(prompt, return_tensors='pt').to(device)

    # Generate response
    with torch.no_grad():
        outputs = model.generate(
            inputs,
            max_length=len(inputs[0]) + max_length,
            temperature=temperature,
            top_p=top_p,
            do_sample=True,
            pad_token_id=tokenizer.eos_token_id,
            num_return_sequences=1
        )

    # Decode response
    response = tokenizer.decode(outputs[0], skip_special_tokens=True)

    # Extract bot response (everything after [Bot])
    if "[Bot]" in response:
        bot_response = response.split("[Bot]")[-1].strip()
        return bot_response

    return response

In [ ]:
# Test with sample questions from each category
test_questions = [
    "How do I validate my startup idea?",  # Mentorship needed
    "What metrics do investors look for?",  # Investment ready
    "My startup is losing customers. What should I do?"  # Needs refinement
]

print(f"=== Qualitative Evaluation ===")
for i, question in enumerate(test_questions, 1):
    print(f"\nSample {i}: {question}")
    response = generate_response(model, tokenizer, question)
    print(f"Response: {response}")
    print("-" * 80)